# Expected R&D spend benchmark

**ML Zoomcamp 2026 midterm project**

**Goal:** predict the R&D spend we'd expect for a company, given its size, sector, profitability and location.
The prediction acts as a benchmark: is a company's reported R&D typical, unusually high or unusually low for its profile?

**Data:** EU Industrial R&D Investment Scoreboard panel (2003–2024), trimmed to the columns below.

| Column | Meaning |
|---|---|
| `company`, `company_name` | Short and long company name |
| `year` | Financial year |
| `new_country`, `region_12_new` | Country and world region of headquarters |
| `icb3_short`, `icb3_name` | Sector: 11 broad groups and 39 detailed sectors |
| `rnd_24` | **Target:** R&D investment, € million (2024 euros) |
| `netsales_24`, `capex_24`, `profit_24`, `mcap_24` | Net sales, capital expenditure, operating profit, market cap (€m, 2024 euros) |
| `emp` | Number of employees |
| `year_inc_new` | Year of incorporation (useful for company age) |

## 1. Setup and load data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('data/scoreboard_panel.csv')
df.head()

## 2. Data cleaning

Things to look at:
- How many rows have a missing target (`rnd_24`)? These can't be used for training.
- Are there zero or negative values in R&D, sales, capex? What should happen to them?
- Missing values in the features: fill (with what?) or drop?
- Hint: `df.isnull().mean()` gives the share missing per column.

## 3. Exploratory data analysis

### 3.1 The target
- Plot the distribution of `rnd_24`. It's very skewed: compare it with `np.log1p(rnd_24)`, as in the car price homework.
- Min, max, median. Which companies are at the extremes?

### 3.2 Features: ranges and distributions
- `describe()` on the numeric columns. Which ones also need a log transform?
- How many companies per sector, region and year?
- How many years does each company appear in? (Hint: `groupby('company')['year'].nunique()`)

### 3.3 R&D intensity by sector
- R&D intensity = R&D ÷ net sales. Compare the median by sector with a bar or box plot.
- Look at the extreme cases (intensity above 100%). What kind of companies are they?

### 3.4 Feature importance
- Correlation of each numeric feature with log R&D (after log-transforming the skewed ones).
- Mutual information for the categorical features (sector, region), as in the churn homework.
- Which features look most useful? Any that look useless?

## 4. Feature engineering and validation split

Ideas for features: log of the money columns, profit margin (profit ÷ sales), company age (year − year of incorporation).

**Two traps to avoid:**
1. **Leakage:** never use anything calculated from R&D spend as a feature.
2. **The same company in train and test:** a random split puts a company's 2019 row in training and its 2020 row in test, which flatters the score.
   Use a split by time (e.g. train ≤ 2019, validation 2020–21, test 2022–24) or by company (`GroupShuffleSplit`).
   Write down which you chose and why.

## 5. Model training

Metric: RMSE on log R&D (and, for interpretation, `np.exp(rmse)` = the typical "factor" the prediction is off by).

### 5.1 Baseline: linear model
- `DictVectorizer` + `Ridge` (or `LinearRegression`). Tune the regularisation strength `alpha`.

### 5.2 Decision tree
- Tune `max_depth` and `min_samples_leaf`, as in HW6.

### 5.3 Random forest
- Tune `n_estimators`, `max_depth`, `min_samples_leaf`.

### 5.4 XGBoost
- Tune `eta`, `max_depth`, `min_child_weight`. Use the validation set to pick the number of rounds.

### 5.5 Comparison
- A small table of validation RMSE for each tuned model. Pick the best one and say why.

## 6. Final model
- Retrain the chosen model on train + validation, evaluate once on the test set.
- Try predicting for one example company and compare with its actual R&D.